In [3]:
import torch
from torch import nn
import torch.profiler
import time
from argparse import Namespace
from model.destseg import DeSTSeg

model = DeSTSeg(dest=True, ed=True).cuda()
model.cuda()
model.eval()

img = torch.randn(1, 3, 224, 224).cuda()


x = torch.randn(1, 3, 224, 224).cuda()
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True,
    with_flops=True
) as prof:
    with torch.no_grad():
        torch.cuda.reset_peak_memory_stats()
        start_time = time.perf_counter()
        output_segmentation, output_de_st, output_de_st_list = model(img)
        end_time = time.perf_counter()
        peak = torch.cuda.max_memory_allocated() / 1024**2
        """
        loss = criterion(output, target)

        # Backward pass
        loss.backward()
        """
    print(f"Peak VRAM: {peak:.1f} MB")

    print("Inference Time: {:.4f} seconds".format(end_time - start_time))

flop_sum = 0
for evt in prof.key_averages():
    if evt.flops is not None:
        flop_sum += evt.flops

print(f"Total FLOPs (Forward + Backward): {flop_sum}")

# Convert to readable format
def format_flops(flops):
    units = ['FLOPs', 'KFLOPs', 'MFLOPs', 'GFLOPs', 'TFLOPs', 'PFLOPs']
    scale = 0
    while flops >= 1000 and scale < len(units) - 1:
        flops /= 1000.0
        scale += 1
    return f"{flops:.2f} {units[scale]}"

print(f"Total FLOPs: {format_flops(flop_sum)}")

STAGE:2026-03-05 10:41:57 2268525:2268525 ActivityProfilerController.cpp:311] Completed Stage: Warm Up
STAGE:2026-03-05 10:41:57 2268525:2268525 ActivityProfilerController.cpp:317] Completed Stage: Collection
STAGE:2026-03-05 10:41:57 2268525:2268525 ActivityProfilerController.cpp:321] Completed Stage: Post Processing
[W util.cpp:528] Warning: Failed to compute flops for op aten::conv2d because it requires padding, stride, and dilation values. (function computeFlops)


Peak VRAM: 2646.0 MB
Inference Time: 0.0759 seconds


[W util.cpp:528] Warning: Failed to compute flops for op aten::conv2d because it requires padding, stride, and dilation values. (function computeFlops)


Total FLOPs (Forward + Backward): 186965823548
Total FLOPs: 186.97 GFLOPs


In [2]:
import torch
from torch import nn
import torch.profiler
import time
from argparse import Namespace
from constant import NORMALIZE_STD, NORMALIZE_MEAN, RESIZE_SHAPE
from model.destseg import DeSTSeg
from data.mvtec_dataset import MVTecDataset
from model.losses import cosine_similarity_loss, focal_loss, l1_loss
import torch.nn.functional as F

mvtec_path = "/home/u0052/disk/datasets/mvtec/"
dtd_path = "/home/u0052/disk/datasets/dtd/images/"

dataset = MVTecDataset(
    is_train=True,
    mvtec_dir=mvtec_path + "bottle" + "/train/good/",
    resize_shape=RESIZE_SHAPE,
    normalize_mean=NORMALIZE_MEAN,
    normalize_std=NORMALIZE_STD,
    dtd_dir=dtd_path,
)


model = DeSTSeg(dest=True, ed=True).cuda()
model.cuda()

sample_batched = next(iter(torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)))

model.train()

x = torch.randn(1, 3, 224, 224).cuda()
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True,
    with_flops=True
) as prof:
    torch.cuda.reset_peak_memory_stats()
    for i in range(2):
        img_origin = sample_batched["img_origin"].cuda()
        img_aug = sample_batched["img_aug"].cuda()
        mask = sample_batched["mask"].cuda()

        if i == 0:
            model.student_net.train()
            model.segmentation_net.eval()
        else:
            model.student_net.eval()
            model.segmentation_net.train()

        output_segmentation, output_de_st, output_de_st_list = model(
            img_aug, img_origin
        )

        mask = F.interpolate(
            mask,
            size=output_segmentation.size()[2:],
            mode="bilinear",
            align_corners=False,
        )
        mask = torch.where(
            mask < 0.5, torch.zeros_like(mask), torch.ones_like(mask)
        )

        cosine_loss_val = cosine_similarity_loss(output_de_st_list)
        focal_loss_val = focal_loss(output_segmentation, mask, gamma=1)
        l1_loss_val = l1_loss(output_segmentation, mask)

        if i == 1:
            total_loss_val = cosine_loss_val
            total_loss_val.backward()
        else:
            total_loss_val = focal_loss_val + l1_loss_val
            total_loss_val.backward()

        peak = torch.cuda.max_memory_allocated() / 1024**2
print(f"Peak VRAM: {peak:.1f} MB")

# ==========================================
# ELABORAZIONE DEI DATI FUORI DAL BLOCCO WITH
# ==========================================
flop_sum = 0
for evt in prof.key_averages():
    if evt.flops is not None:
        flop_sum += evt.flops

print(f"Total FLOPs (Forward + Backward): {flop_sum}")

# Funzione per formattare l'output
def format_flops(flops):
    units = ['FLOPs', 'KFLOPs', 'MFLOPs', 'GFLOPs', 'TFLOPs', 'PFLOPs']
    scale = 0
    while flops >= 1000 and scale < len(units) - 1:
        flops /= 1000.0
        scale += 1
    return f"{flops:.2f} {units[scale]}"

print(f"Total FLOPs: {format_flops(flop_sum)}")

INFO:2026-03-05 10:43:22 2269006:2269006 init.cpp:149] If you see CUPTI_ERROR_INSUFFICIENT_PRIVILEGES, refer to https://developer.nvidia.com/nvidia-development-tools-solutions-err-nvgpuctrperm-cupti
STAGE:2026-03-05 10:43:22 2269006:2269006 ActivityProfilerController.cpp:311] Completed Stage: Warm Up
STAGE:2026-03-05 10:43:25 2269006:2269006 ActivityProfilerController.cpp:317] Completed Stage: Collection
STAGE:2026-03-05 10:43:25 2269006:2269006 ActivityProfilerController.cpp:321] Completed Stage: Post Processing
[W util.cpp:528] Warning: Failed to compute flops for op aten::conv2d because it requires padding, stride, and dilation values. (function computeFlops)
[W util.cpp:528] Warning: Failed to compute flops for op aten::conv2d because it requires padding, stride, and dilation values. (function computeFlops)
[W util.cpp:528] Warning: Failed to compute flops for op aten::conv2d because it requires padding, stride, and dilation values. (function computeFlops)
[W util.cpp:528] Warning:

Peak VRAM: 3241.3 MB
Total FLOPs (Forward + Backward): 976830541827
Total FLOPs: 976.83 GFLOPs
